In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/johnsmith88/heart-disease-dataset/heart.csv


In [3]:
df = pd.read_csv('/kaggle/input/datasets/johnsmith88/heart-disease-dataset/heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [5]:
df.shape

(1025, 14)

In [7]:
df.isnull().mean()

age         0.0
sex         0.0
cp          0.0
trestbps    0.0
chol        0.0
fbs         0.0
restecg     0.0
thalach     0.0
exang       0.0
oldpeak     0.0
slope       0.0
ca          0.0
thal        0.0
target      0.0
dtype: float64

In [6]:
from sklearn.model_selection import train_test_split
X = df.drop(columns = ['target'])
y = df.target

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state = 42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((820, 13), (205, 13), (820,), (205,))

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

import warnings
warnings.filterwarnings('ignore')

In [9]:
logistic_reg = LogisticRegression()
random_forest = RandomForestClassifier()
gradientboost = GradientBoostingClassifier()
svc = SVC()

In [10]:
logistic_reg.fit(X_train, y_train)
random_forest.fit(X_train, y_train)
gradientboost.fit(X_train, y_train)
svc.fit(X_train, y_train)

SVC()

In [11]:
from sklearn.metrics import accuracy_score

logistic_pred = logistic_reg.predict(X_test)
accuracy_score(y_test, logistic_pred)

0.7804878048780488

In [12]:
random_forest_pred = random_forest.predict(X_test)
accuracy_score(y_test, random_forest_pred)

0.9853658536585366

In [13]:
gradientboost_pred = gradientboost.predict(X_test)
accuracy_score(y_test, gradientboost_pred)

0.9317073170731708

In [14]:
svc_pred = svc.predict(X_test)
accuracy_score(y_test, svc_pred)

0.6829268292682927

### RandomForest With Hyperparameter Tuning

In [22]:
random_forest_2 = RandomForestClassifier(max_samples = 0.75, random_state = 42)
random_forest_2.fit(X_train, y_train)
accuracy_score(y_test, random_forest_2.predict(X_test))

0.9853658536585366

In [27]:
from sklearn.model_selection import cross_val_score
np.mean(cross_val_score(
    RandomForestClassifier(max_samples =0.75),
    X,
    y,
    cv=10,
    scoring='accuracy'
))

np.float64(0.9970588235294118)

## GridSearchCV

In [28]:
n_estimators = [60,100,200]
max_features = [0.5, 0.6, 1.0]
max_samples = [0.5, 0.6,1.0]
max_depth = [2,7,None]

In [36]:
param_grid = {
    'n_estimators': n_estimators,
    'max_samples': max_samples,
    'max_features': max_features,
    'max_depth': max_depth
}

In [37]:
grid_search_cv = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,
    n_jobs = -1,
    verbose = 1
)

In [38]:
grid_search_cv.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'max_depth': [2, 7, None],
                         'max_features': [0.5, 0.6, 1.0],
                         'max_samples': [0.5, 0.6, 1.0],
                         'n_estimators': [60, 100, 200]},
             verbose=1)

In [39]:
grid_search_cv.best_params_

{'max_depth': None,
 'max_features': 0.5,
 'max_samples': 1.0,
 'n_estimators': 100}

In [40]:
grid_search_cv.best_score_

np.float64(0.9817073170731707)

In [41]:
model = grid_search_cv.best_estimator_

In [42]:
accuracy_score(y_test, model.predict(X_test))

0.9853658536585366

In [43]:
X_test.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
527,62,0,0,124,209,0,1,163,0,0.0,2,0,2
359,53,0,2,128,216,0,0,115,0,0.0,2,0,0
447,55,1,0,160,289,0,0,145,1,0.8,1,1,3
31,50,0,1,120,244,0,1,162,0,1.1,2,0,2
621,48,1,0,130,256,1,0,150,1,0.0,2,2,3


In [44]:
y_test.head()

527    1
359    1
447    0
31     1
621    0
Name: target, dtype: int64

In [45]:
model.predict(np.array([62,	0,	0,	124,	209,	0,	1,	163,	0,	0.0,	2,	0,	2]).reshape(1,-1))

array([1])

### RandomizedSearchCV

In [46]:
# Number of trees in random forest
n_estimators = [20,60,100,120]

# Number of features to consider at every split
max_features = [0.2,0.6,1.0]

# Maximum number of levels in tree
max_depth = [2,8,None]

# Number of samples
max_samples = [0.5,0.75,1.0]

# Bootstrap samples
bootstrap = [True,False]

# Minimum number of samples required to split a node
min_samples_split = [2, 5]

# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2]

In [47]:
param_distribution = {
    'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
              'max_samples':max_samples,
              'bootstrap':bootstrap,
              'min_samples_split':min_samples_split,
              'min_samples_leaf':min_samples_leaf
}

In [48]:
randomized_search_cv = RandomizedSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=param_distribution,
    cv=5,
    n_jobs=-1,
    verbose=1
)
randomized_search_cv.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [2, 8, None],
                                        'max_features': [0.2, 0.6, 1.0],
                                        'max_samples': [0.5, 0.75, 1.0],
                                        'min_samples_leaf': [1, 2],
                                        'min_samples_split': [2, 5],
                                        'n_estimators': [20, 60, 100, 120]},
                   verbose=1)

In [49]:
randomized_search_cv.best_params_

{'n_estimators': 120,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 1.0,
 'max_features': 0.6,
 'max_depth': 8,
 'bootstrap': True}

In [50]:
randomized_search_cv.best_score_

np.float64(0.9792682926829268)

In [52]:
model_2 = randomized_search_cv.best_estimator_

In [53]:
accuracy_score(y_test, model_2.predict(X_test))

0.9853658536585366

In [54]:
model_2.predict(np.array([62,	0,	0,	124,	209,	0,	1,	163,	0,	0.0,	2,	0,	2]).reshape(1,-1))

array([1])